## Model Logging with MLflow

### Setup the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Connect to Azure ML Workspace

In [ ]:
import azureml.core
from azureml.core import Workspace

# Load the workspace from the saved config file
ws = Workspace.from_config()
print('Ready to use Azure ML {} to work with {}'.format(azureml.core.VERSION, ws.name))

### Fetch the Diabetes Dataset from the Data Asset

In [ ]:
from azureml.core import Workspace, Dataset

dataset = Dataset.get_by_name(ws, name='diabetes dataset')
diabetes_df = dataset.to_pandas_dataframe()

In [ ]:
display(diabetes_df)

### Prepare the Testing and Training Data with Normalization and Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pandas as pd

numericFeatures = [
            "Pregnancies",
            "Glucose",
            "BloodPressure",
            "SkinThickness",
            "Insulin",
            "BMI",
            "DiabetesPedigreeFunction",
            "Age"
]

# Extract the feature matrix (similar to VectorAssembler)
numericFeaturesVector = diabetes_df[numericFeatures]

# Apply MinMax Scaling
scaler = MinMaxScaler()

normalizedFeatures = scaler.fit_transform(numericFeaturesVector)

# Create a scaled dataframe
scaledData = pd.DataFrame(
    normalizedFeatures,
    columns=numericFeatures
)

# Add the label column
scaledData["Outcome"] = diabetes_df["Outcome"]
train, test = train_test_split(scaledData, test_size=0.3, random_state=42)

### Creating Mlflow Experiment Function and Logging Model Runs

In [ ]:
import mlflow
mlflow.set_experiment("mlflow-experiment")

In [ ]:
def train_diabetes_model(maxIterations, regularization):
        import mlflow
        import mlflow.sklearn
        from mlflow.models import infer_signature
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import MinMaxScaler
        from sklearn.metrics import accuracy_score, recall_score, precision_score
        import pandas as pd
        import time
        from sklearn.model_selection import train_test_split
    
    
        # Start an MLflow run  
        mlflow.start_run()

        # make up the testing and the training datasets
        X_train = train[numericFeatures]
        Y_train = train['Outcome']
        X_test = test[numericFeatures]
        Y_test = test["Outcome"]
        
        # Log hyperparameters with MLflow Runs
        print("Training Logistic Regression model...")
        mlflow.log_param('maxIter', maxIterations)
        mlflow.log_param('regParam', regularization)

        model = LogisticRegression(
            max_iter=maxIterations,
            C=1/regularization
        )
        model.fit(X_train, Y_train)

        # Evaluate the model
        y_pred = model.predict(X_test)
        
        accuracy = accuracy_score(Y_test, y_pred)
        recall = recall_score(Y_test, y_pred, average='weighted')
        precision = precision_score(Y_test, y_pred, average='weighted')
        
        # log evaluation metrics with Mlflow runs
        print("accuracy: %s" % accuracy)
        print("weightedRecall: %s" % recall)
        print("weightedPrecision: %s" % precision)
        
        mlflow.log_metric('accuracy', accuracy)
        mlflow.log_metric('weightedRecall', recall)
        mlflow.log_metric('weightedPrecision', precision)

        # Create signature
        input_example = pd.DataFrame([{
            "Pregnancies": 8, "Glucose": 85, "BloodPressure": 65, "SkinThickness": 29,
            "Insulin": 0, "BMI": 26.6, "DiabetesPedigreeFunction": 0.672, "Age": 34
        }])
        output_example = pd.DataFrame({"prediction": [1]})
        signature = infer_signature(input_example, output_example)
   
        # Log the sklearn model with signature
        unique_model_name = "diabetes_sklearn_" + str(int(time.time()))
        
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=input_example
        )
   
        print("Experiment run complete with sklearn model.")

        mlflow.end_run()

### Calling the Mlflow Experiment Function with Different Hyperparameters

In [ ]:
train_diabetes_model(5, 0.5)

In [ ]:
train_diabetes_model(10, 0.2)